In [0]:
# 08_Audit_Log_Function

import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

audit_schema = StructType([
    StructField("config_id", StringType(), True),
    StructField("log_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("layer_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("record_count", IntegerType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("error_msg", StringType(), True),
])

def log_audit(layer_name, table_name=None, status="SUCCESS", start_time=None, error_msg=None):
    log_id = str(uuid.uuid4())
    record_count = spark.table(table_name).count() if (status == "SUCCESS" and table_name) else 0

    row = [(
        config_id,
        log_id,
        pipeline_name,
        layer_name,
        status,
        record_count,
        start_time,
        error_msg
    )]

    df_audit = spark.createDataFrame(row, schema=audit_schema)
    df_audit.write.format("delta").mode("append").saveAsTable(audit_table)

    print(f"Audit logged -> layer: {layer_name}, status: {status}, rows: {record_count}")